[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hg1112/learning-monorepo/blob/main/challenges/mle/notebooks/week2_ads_ml/day7_rtb_simulator.ipynb)

# Week 2, Day 7 — RTB Auction Simulator
**Study Plan:** Pinterest Staff MLE Prep | Week 2 Capstone

**Paper:** [Real-Time Bidding Algorithms for Performance-Based Display Ad Allocation — Zhang et al. (2012)](https://dl.acm.org/doi/10.1145/2339530.2339604)

**Goal:** Build a working RTB simulator that integrates:
1. A calibrated CTR model (from Day 5)
2. A PID budget pacer (from Day 6)
3. A first-price and second-price auction engine
4. Optimal bidding with the Lagrangian approach

**Week 2 Deliverable:** Working RTB simulator with calibrated CTR model + PID budget pacer.

## 1. Concept Recap

### The RTB Decision Pipeline (per impression, <100ms)
```
Bid Request → pCTR prediction → eCPM = pCTR × CPC → Pacing check → Bid shading → Submit
```

### Auction Types
- **Second-price:** Winner pays 2nd-highest bid. Dominant strategy = bid true value `v = pCTR × CPC`.
- **First-price:** Winner pays their own bid. Must shade: `b* < v` to avoid winner's curse.

### Lagrangian Budget-Constrained Bidding
Maximize clicks s.t. total spend ≤ B:
```
b*_t = pCTR_t / λ
```
where λ (the budget shadow price) is adjusted by the PID pacer:
- Overspending → λ↑ → bids drop
- Underspending → λ↓ → bids rise

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from dataclasses import dataclass, field
from typing import List, Tuple, Optional

np.random.seed(42)
plt.style.use('ggplot')
print('Libraries loaded.')

## 2. Synthetic Auction Stream

We simulate a day of bid requests. Each request has:
- A **true CTR** (drawn from a Beta distribution — calibrated to ~2% mean, realistic for display ads)
- A **floor price** (SSP minimum)
- A **market price** (what a competitor would bid — log-normal, typical for RTB)

In [ ]:
@dataclass
class BidRequest:
    impression_id: int
    true_ctr: float        # ground truth (unknown to bidder)
    noisy_ctr: float       # model prediction (pCTR)
    floor_price: float     # SSP minimum bid (CPM)
    market_price: float    # highest competitor bid (CPM) — revealed only if we win
    user_quality: float    # 0-1 score (high-value user = higher true CTR)

def generate_auction_stream(n_impressions: int = 10_000, cpc_value: float = 1.0) -> List[BidRequest]:
    """
    Simulate a day's worth of bid requests.
    
    CTR distribution: Beta(0.5, 24.5) → mean ~2%, right-skewed (most impressions low value)
    Market price: LogNormal(0, 0.6) in CPM cents → realistic RTB price distribution
    Floor price: Uniform(0.1, 0.5) CPM
    """
    user_quality = np.random.beta(2, 5, n_impressions)           # 0-1, mean ~0.28
    true_ctrs = np.random.beta(0.5 + 2 * user_quality,           # quality boosts CTR
                                24.5 - 2 * user_quality)
    
    # Model prediction adds noise: pCTR = true_ctr * exp(noise)
    noise = np.random.normal(0, 0.3, n_impressions)
    noisy_ctrs = np.clip(true_ctrs * np.exp(noise), 0.001, 0.5)
    
    floor_prices = np.random.uniform(0.1, 0.5, n_impressions)    # CPM
    
    # Market (competitor) prices: log-normal, correlated with user quality
    market_prices = np.random.lognormal(
        mean=0.5 + 0.8 * user_quality,
        sigma=0.6,
        size=n_impressions
    )
    
    return [
        BidRequest(
            impression_id=i,
            true_ctr=true_ctrs[i],
            noisy_ctr=noisy_ctrs[i],
            floor_price=floor_prices[i],
            market_price=market_prices[i],
            user_quality=user_quality[i],
        )
        for i in range(n_impressions)
    ]

stream = generate_auction_stream(n_impressions=10_000)
ctrs = [r.true_ctr for r in stream]
print(f'Generated {len(stream):,} bid requests')
print(f'True CTR: mean={np.mean(ctrs):.3f}, median={np.median(ctrs):.3f}, p95={np.percentile(ctrs, 95):.3f}')
print(f'Market price: mean={np.mean([r.market_price for r in stream]):.2f} CPM')

## 3. Auction Engine (First-Price and Second-Price)

The exchange runs the auction after collecting all bids. We model:
- **One DSP** (us) competing against **one representative market bid**
- Win condition: our bid > market_price AND our bid > floor_price
- **Second-price:** we pay `max(floor_price, market_price) + ε`
- **First-price:** we pay our own bid

In [ ]:
@dataclass
class AuctionResult:
    won: bool
    clearing_price: float   # what we actually pay (CPM)
    our_bid: float
    
def run_auction(
    request: BidRequest,
    our_bid: float,
    auction_type: str = 'first_price',   # 'first_price' | 'second_price'
) -> AuctionResult:
    """
    Simulate a single RTB auction.
    
    Win condition: our_bid > floor AND our_bid > market_price
    """
    effective_floor = max(request.floor_price, request.market_price)
    won = our_bid > effective_floor
    
    if not won:
        return AuctionResult(won=False, clearing_price=0.0, our_bid=our_bid)
    
    if auction_type == 'second_price':
        # Pay the highest losing bid + epsilon
        clearing_price = effective_floor + 0.01
    else:  # first_price
        # Pay your own bid
        clearing_price = our_bid
    
    return AuctionResult(won=True, clearing_price=clearing_price, our_bid=our_bid)

## 4. PID Budget Pacer

Recap from Day 6. The pacer controls **λ** (the Lagrange multiplier) to track a target spend rate.

```
target_spend_rate = total_budget / total_time_slots
error = actual_spend - target_spend  (positive = overspending)
λ(t+1) = λ(t) + Kp * error + Ki * integral_error + Kd * derivative_error
```

**λ** is the bid multiplier: `bid = eCPM / λ`. λ=1 means bid at full value; λ>1 means discounted bids.

In [ ]:
@dataclass
class PIDPacer:
    budget: float              # total daily budget (dollars)
    n_slots: int               # number of time slots in the day
    kp: float = 0.1            # proportional gain
    ki: float = 0.01           # integral gain
    kd: float = 0.05           # derivative gain
    
    lam: float = field(default=1.0, init=False)        # current λ
    spent: float = field(default=0.0, init=False)      # cumulative spend
    slot: int = field(default=0, init=False)
    _integral: float = field(default=0.0, init=False)
    _prev_error: float = field(default=0.0, init=False)
    
    history: List = field(default_factory=list, init=False)
    
    def target_spend_at_slot(self) -> float:
        """Ideal cumulative spend at current slot (linear pacing)."""
        return self.budget * (self.slot / self.n_slots)
    
    def update(self, amount_spent: float) -> float:
        """Update pacer after a slot. Returns new λ."""
        self.spent += amount_spent
        self.slot += 1
        
        target = self.target_spend_at_slot()
        error = self.spent - target   # positive = overspending
        
        self._integral += error
        derivative = error - self._prev_error
        self._prev_error = error
        
        # λ adjustment: overspend → raise λ → lower bids
        delta = self.kp * error + self.ki * self._integral + self.kd * derivative
        self.lam = max(0.1, self.lam + delta)   # floor at 0.1 to avoid zero bids
        
        self.history.append({
            'slot': self.slot,
            'spent': self.spent,
            'target': target,
            'lam': self.lam,
            'error': error,
        })
        return self.lam
    
    def remaining_budget(self) -> float:
        return max(0.0, self.budget - self.spent)
    
    def is_budget_exhausted(self) -> bool:
        return self.spent >= self.budget

## 5. Bidding Strategies

We'll compare four strategies:

| Strategy | Bid Formula | Notes |
|----------|-------------|-------|
| **Truth-telling** | `b = pCTR × CPC` | Optimal for 2nd-price, naïve for 1st-price |
| **Fixed fraction** | `b = α × pCTR × CPC` | Constant shade factor α |
| **Lagrangian** | `b = pCTR × CPC / λ` | λ from PID pacer |
| **Optimal shade** | `b = v × (1 - 1/n)` | Theoretical optimal for n symmetric bidders |

In [ ]:
def compute_bid(
    request: BidRequest,
    strategy: str,
    cpc_value: float,          # advertiser's max CPC ($)
    lam: float = 1.0,          # from PID pacer
    alpha: float = 0.7,        # for fixed_fraction
    n_competitors: int = 5,    # for optimal_shade
) -> float:
    """
    Compute bid in CPM units.
    
    eCPM = pCTR × CPC × 1000   (convert CPC to CPM)
    """
    ecpm = request.noisy_ctr * cpc_value * 1000   # eq. value in CPM
    
    if strategy == 'truth_telling':
        return ecpm
    
    elif strategy == 'fixed_fraction':
        # Shade by fixed factor — common naive approach
        return alpha * ecpm
    
    elif strategy == 'lagrangian':
        # Zhang et al. (2012): b* = v / λ — λ from PID pacer
        return ecpm / lam
    
    elif strategy == 'optimal_shade':
        # Theoretical optimum for first-price with n symmetric bidders:
        # b* = v * (n-1)/n
        shade = (n_competitors - 1) / n_competitors
        return shade * ecpm
    
    else:
        raise ValueError(f'Unknown strategy: {strategy}')

## 6. Full Simulation Runner

Simulate a full day: impressions arrive in `n_slots` time buckets. Each slot we:
1. Process all impressions in that slot
2. Compute bids using our strategy
3. Run auctions
4. Tally spend
5. Update PID pacer → new λ for next slot

In [ ]:
@dataclass
class SimulationResult:
    strategy: str
    auction_type: str
    total_impressions: int
    total_wins: int
    total_clicks: int          # sampled from true CTR
    total_spend: float
    budget: float
    cpc_value: float
    spend_curve: List[float]   # cumulative spend per slot
    target_curve: List[float]  # target spend per slot
    lam_curve: List[float]     # λ per slot
    
    @property
    def win_rate(self) -> float:
        return self.total_wins / self.total_impressions
    
    @property
    def avg_cpc(self) -> float:
        return self.total_spend / self.total_clicks if self.total_clicks > 0 else 0
    
    @property
    def budget_utilization(self) -> float:
        return self.total_spend / self.budget
    
    @property
    def roi(self) -> float:
        """Clicks per dollar spent."""
        return self.total_clicks / self.total_spend if self.total_spend > 0 else 0


def run_simulation(
    stream: List[BidRequest],
    strategy: str,
    auction_type: str = 'first_price',
    budget: float = 50.0,      # daily budget in dollars
    cpc_value: float = 1.0,    # advertiser max CPC
    n_slots: int = 100,        # time buckets in a day
    **strategy_kwargs
) -> SimulationResult:
    """
    Run a full day simulation.
    Impressions are divided evenly across n_slots.
    """
    pacer = PIDPacer(budget=budget, n_slots=n_slots)
    slot_size = len(stream) // n_slots
    
    total_wins = 0
    total_clicks = 0
    spend_curve = []
    target_curve = []
    lam_curve = []
    
    for slot in range(n_slots):
        if pacer.is_budget_exhausted():
            # Fill remaining slots with zeros
            for _ in range(slot, n_slots):
                pacer.update(0)
                spend_curve.append(pacer.spent)
                target_curve.append(pacer.target_spend_at_slot())
                lam_curve.append(pacer.lam)
            break
        
        slot_impressions = stream[slot * slot_size : (slot + 1) * slot_size]
        slot_spend = 0.0
        
        for req in slot_impressions:
            bid = compute_bid(
                req,
                strategy=strategy,
                cpc_value=cpc_value,
                lam=pacer.lam,
                **strategy_kwargs
            )
            
            result = run_auction(req, bid, auction_type=auction_type)
            
            if result.won:
                cost = result.clearing_price / 1000  # CPM → dollars
                if pacer.spent + slot_spend + cost <= budget:
                    total_wins += 1
                    slot_spend += cost
                    # Sample actual click from true CTR
                    if np.random.random() < req.true_ctr:
                        total_clicks += 1
        
        pacer.update(slot_spend)
        spend_curve.append(pacer.spent)
        target_curve.append(pacer.target_spend_at_slot())
        lam_curve.append(pacer.lam)
    
    return SimulationResult(
        strategy=strategy,
        auction_type=auction_type,
        total_impressions=len(stream),
        total_wins=total_wins,
        total_clicks=total_clicks,
        total_spend=pacer.spent,
        budget=budget,
        cpc_value=cpc_value,
        spend_curve=spend_curve,
        target_curve=target_curve,
        lam_curve=lam_curve,
    )

## 7. Run All Strategies and Compare

In [ ]:
BUDGET = 50.0
CPC_VALUE = 1.0

configs = [
    ('truth_telling',   'second_price', {}),
    ('truth_telling',   'first_price',  {}),
    ('fixed_fraction',  'first_price',  {'alpha': 0.7}),
    ('optimal_shade',   'first_price',  {'n_competitors': 5}),
    ('lagrangian',      'first_price',  {}),
]

results = {}
for strategy, auction_type, kwargs in configs:
    key = f'{strategy}/{auction_type}'
    results[key] = run_simulation(
        stream,
        strategy=strategy,
        auction_type=auction_type,
        budget=BUDGET,
        cpc_value=CPC_VALUE,
        **kwargs
    )
    r = results[key]
    print(f'{key:40s}  wins={r.total_wins:4d}  clicks={r.total_clicks:4d}  '
          f'spend=${r.total_spend:.2f}  util={r.budget_utilization:.1%}  '
          f'avg_cpc=${r.avg_cpc:.3f}  roi={r.roi:.2f}')

## 8. Visualization: Spend Curves + Performance

In [ ]:
fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

colors = ['#2196F3', '#F44336', '#FF9800', '#4CAF50', '#9C27B0']

# --- Spend curves ---
ax1 = fig.add_subplot(gs[0, :])
for (key, r), color in zip(results.items(), colors):
    ax1.plot(r.spend_curve, label=key, color=color, linewidth=2)
ax1.plot(results[list(results.keys())[0]].target_curve,
         '--', color='black', linewidth=2, label='target (ideal linear pacing)', alpha=0.7)
ax1.axhline(BUDGET, color='red', linestyle=':', alpha=0.5, label=f'budget=${BUDGET}')
ax1.set_xlabel('Time Slot')
ax1.set_ylabel('Cumulative Spend ($)')
ax1.set_title('Budget Spend Curves — All Strategies')
ax1.legend(fontsize=9, ncol=3)

# --- Win rates ---
ax2 = fig.add_subplot(gs[1, 0])
keys = list(results.keys())
win_rates = [results[k].win_rate for k in keys]
bars = ax2.barh(keys, win_rates, color=colors)
ax2.set_xlabel('Win Rate')
ax2.set_title('Win Rate by Strategy')
ax2.set_xlim(0, max(win_rates) * 1.3)
for bar, v in zip(bars, win_rates):
    ax2.text(v + 0.002, bar.get_y() + bar.get_height()/2,
             f'{v:.1%}', va='center', fontsize=9)

# --- Clicks ---
ax3 = fig.add_subplot(gs[1, 1])
clicks = [results[k].total_clicks for k in keys]
bars = ax3.barh(keys, clicks, color=colors)
ax3.set_xlabel('Total Clicks')
ax3.set_title('Clicks by Strategy')
for bar, v in zip(bars, clicks):
    ax3.text(v + 0.5, bar.get_y() + bar.get_height()/2,
             str(v), va='center', fontsize=9)

# --- Budget utilization ---
ax4 = fig.add_subplot(gs[1, 2])
utils = [results[k].budget_utilization for k in keys]
bars = ax4.barh(keys, utils, color=colors)
ax4.axvline(1.0, color='red', linestyle='--', alpha=0.7, label='100% utilized')
ax4.set_xlabel('Budget Utilization')
ax4.set_title('Budget Utilization by Strategy')
for bar, v in zip(bars, utils):
    ax4.text(v + 0.01, bar.get_y() + bar.get_height()/2,
             f'{v:.1%}', va='center', fontsize=9)

plt.suptitle('RTB Auction Simulator — Strategy Comparison', fontsize=14, fontweight='bold')
plt.savefig('rtb_simulation_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved.')

## 9. Deep Dive: The Lagrangian Bidder

Let's trace how λ evolves for the Lagrangian strategy and what it means economically.

In [ ]:
r_lag = results['lagrangian/first_price']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# λ curve
ax = axes[0]
ax.plot(r_lag.lam_curve, color='#9C27B0', linewidth=2)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.7, label='λ=1 (bid full value)')
ax.set_xlabel('Time Slot')
ax.set_ylabel('λ (bid discount factor)')
ax.set_title('Lagrange Multiplier λ over Day')
ax.legend()

# Spend tracking
ax = axes[1]
ax.plot(r_lag.spend_curve, label='actual spend', color='#9C27B0', linewidth=2)
ax.plot(r_lag.target_curve, '--', color='black', linewidth=2, label='target spend', alpha=0.7)
ax.fill_between(range(len(r_lag.spend_curve)),
                r_lag.spend_curve, r_lag.target_curve,
                alpha=0.15, color='#9C27B0')
ax.set_xlabel('Time Slot')
ax.set_ylabel('Cumulative Spend ($)')
ax.set_title('Lagrangian Spend Tracking')
ax.legend()

plt.tight_layout()
plt.show()

print(f"λ range: [{min(r_lag.lam_curve):.3f}, {max(r_lag.lam_curve):.3f}]")
print(f"λ mean: {np.mean(r_lag.lam_curve):.3f}")
print(f"Spend tracking RMSE: ${np.sqrt(np.mean([(s-t)**2 for s,t in zip(r_lag.spend_curve, r_lag.target_curve)])):.2f}")

## 10. Winner's Curse Analysis

**Winner's curse:** In first-price auctions, you tend to win when you overbid relative to the market. This means the average clearing price you pay skews high — you systematically overpay.

Let's measure how much overpayment occurs for each strategy.

In [ ]:
def measure_winners_curse(stream: List[BidRequest], strategy: str,
                           auction_type: str, n_samples: int = 2000,
                           cpc_value: float = 1.0, **kwargs) -> dict:
    """Measure overpayment relative to true value on won impressions."""
    sample = stream[:n_samples]
    overpayments = []
    
    for req in sample:
        true_ecpm = req.true_ctr * cpc_value * 1000
        bid = compute_bid(req, strategy=strategy, cpc_value=cpc_value, **kwargs)
        result = run_auction(req, bid, auction_type=auction_type)
        
        if result.won:
            # Overpayment = clearing_price - true_value
            # Positive = paid more than the impression was worth
            overpayments.append(result.clearing_price - true_ecpm)
    
    return {
        'mean_overpay': np.mean(overpayments) if overpayments else 0,
        'median_overpay': np.median(overpayments) if overpayments else 0,
        'pct_overpaid': np.mean([o > 0 for o in overpayments]) if overpayments else 0,
        'n_wins': len(overpayments),
    }

print(f"{'Strategy':<40} {'mean_overpay':>12} {'median_overpay':>14} {'% overpaid':>10} {'n_wins':>8}")
print('-' * 90)

curse_configs = [
    ('truth_telling',  'second_price', {}),
    ('truth_telling',  'first_price',  {}),
    ('fixed_fraction', 'first_price',  {'alpha': 0.7}),
    ('optimal_shade',  'first_price',  {'n_competitors': 5}),
    ('lagrangian',     'first_price',  {}),
]

for strategy, auction_type, kwargs in curse_configs:
    key = f'{strategy}/{auction_type}'
    m = measure_winners_curse(stream, strategy, auction_type, **kwargs)
    print(f'{key:<40} {m["mean_overpay"]:>+12.3f} {m["median_overpay"]:>+14.3f} '
          f'{m["pct_overpaid"]:>10.1%} {m["n_wins"]:>8}')

## 11. Pinterest-Scale Context

What would this look like in Pinterest's production system?

| Dimension | This Simulation | Pinterest Production |
|-----------|----------------|----------------------|
| Impressions/day | 10,000 | ~50B+ ad auctions/day |
| Advertisers | 1 | ~5M+ advertisers |
| Bid latency | N/A (offline) | <100ms end-to-end |
| CTR model | Noisy Beta | Two-tower + DCN-v2 |
| Pacing | PID with 100 slots | Intra-second feedback |
| Auction type | First-price | First-price (2021+) |

**Key differences in production:**
1. **Multi-advertiser pacing:** λ is maintained *per advertiser campaign*, not globally
2. **Autobid mode:** Pinterest's system optimizes λ automatically (Performance+ Bids)
3. **Warm start:** λ is initialized from yesterday's converged value
4. **Floor price modeling:** Floor prices are themselves ML-predicted, not uniform random
5. **Win/loss feedback:** Real-time feedback updates CTR models and win-rate models

In [ ]:
# Summary table
rows = []
for key, r in results.items():
    rows.append({
        'Strategy': key,
        'Win Rate': f'{r.win_rate:.1%}',
        'Total Clicks': r.total_clicks,
        'Total Spend': f'${r.total_spend:.2f}',
        'Budget Util': f'{r.budget_utilization:.1%}',
        'Avg CPC': f'${r.avg_cpc:.3f}',
        'ROI (clicks/$)': f'{r.roi:.2f}',
    })

df = pd.DataFrame(rows)
print('\n=== FINAL RESULTS SUMMARY ===')
print(df.to_string(index=False))

## 12. Your Turn — Guided Experiments

Work through these to deepen your understanding:

### Experiment A: Tight Budget
Run the Lagrangian strategy with `budget=10.0` (very tight) and `budget=200.0` (very loose). How does λ behave differently? What does that tell you about the pacing system?

```python
r_tight = run_simulation(stream, strategy='lagrangian', auction_type='first_price',
                         budget=10.0, cpc_value=1.0)
r_loose = run_simulation(stream, strategy='lagrangian', auction_type='first_price',
                         budget=200.0, cpc_value=1.0)
# Plot their lam_curves side by side
```

### Experiment B: PID Tuning
What happens when you set `ki=0` (no integral term)? Does the pacer accumulate steady-state error? Try `kp=0.5` — does it oscillate?

### Experiment C: Noisy CTR Model
In `generate_auction_stream`, the noise is `Normal(0, 0.3)`. Increase it to `0.8` to simulate a bad CTR model. Which strategy is most robust to model noise?

### Experiment D: Market Price Distribution Shift
Simulate a competitor raising their bids (shift the log-normal mean from `0.5` to `1.5`). How does win rate change for each strategy? What would you do in production?

### Experiment E: Floor Price Spikes
Set `floor_prices = np.random.uniform(1.0, 3.0, n)` (high floors). Which strategies fail to win any impressions? Why? How would bid shading need to adapt?

In [ ]:
# Your experiments here


## 13. Week 2 Deliverable — Self-Assessment

You've now built the full Week 2 deliverable: **RTB simulator + calibrated CTR model + PID budget pacer**.

Can you answer these interview questions without notes?

1. **Why did the industry switch from second-price to first-price auctions around 2019-2020?**
2. **What is bid shading and why is it necessary in first-price auctions?**
3. **Derive the optimal bid for a budget-constrained advertiser. What does λ represent economically?**
4. **Your CTR model AUC is 0.80 but your ads underspend the budget. What's wrong?**
5. **How would you A/B test a new bidding algorithm in production without hurting revenue?**

---
*Next up: Week 3, Day 1 — Feature Pipeline Design*